In [ ]:
"""
Read and visualize a VizieR-format .dat catalogue (e.g. California Legacy Survey)
using astropy (to correctly parse the fixed-width CDS format) and pandas
(for exploration and plotting).

VizieR .dat files are fixed-width, with column definitions living in a
separate ReadMe file -- NOT in the .dat file itself, and there's no header
row. astropy.io.ascii with format='cds' parses the ReadMe automatically and
returns a proper Table with column names, units, and descriptions attached.
"""
import astropy.units as u
import os
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import ascii as ascii_io
import argparse
import re
import sys
from pathlib import Path
import numpy as np
from scipy import stats as sstats
from scipy.stats import gaussian_kde
import seaborn as sns

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 10,
    "axes.labelsize": 15,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.direction": "in",
    "ytick.direction": "in",
    # major ticks
    "xtick.major.size": 12,
    "ytick.major.size": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    # minor ticks
    "xtick.minor.size": 8,
    "ytick.minor.size": 8,
    "xtick.minor.visible": True,   # minor ticks are hidden by default — need this to actually see them
    "ytick.minor.visible": True,

})

COLORS = {
    "confirmed": "#2a6f97",
    "candidate": "#e07a5f",
    "kepler": "#81b29a",
    "tess": "#f2cc8f",
    "super_earth": "#803d65",
    "cold_giant": "#4dd3ee",
    "both": "#6a4c93",
    "other": "#c9c9c9",
}

# ------------------------------------------------------------------
# The catalogue
# ------------------------------------------------------------------
DATA_DIR = "/Users/claudia/Documents/PostDoc/CLS_catalogue/J_ApJS_255_8"   # folder containing the CLS catalogue
DAT_FILE = "planets.dat"             # the file I want to read (the .dat file from VizieR)
README_FILE = "ReadMe"             # the CDS ReadMe that came with the .dat

dat_path = os.path.join(DATA_DIR, DAT_FILE)
readme_path = os.path.join(DATA_DIR, README_FILE)

######## to fix the ReadMe file, if needed (e.g. missing space after colon) ########
readme_path = "/Users/claudia/Documents/PostDoc/CLS_catalogue/J_ApJS_255_8/ReadMe"
readme_fixed_path = "/Users/claudia/Documents/PostDoc/CLS_catalogue/J_ApJS_255_8/ReadMe_fixed"

with open(readme_path, "r") as f:
    content = f.read()

content = content.replace(
    "Byte-by-byte Description of file:table7.dat",
    "Byte-by-byte Description of file: table7.dat"
).replace(
    "Byte-by-byte Description of file:apf50.dat",
    "Byte-by-byte Description of file: apf50.dat"
).replace(
    "Byte-by-byte Description of file:planets.dat",
    "Byte-by-byte Description of file: planets.dat"
)

with open(readme_fixed_path, "w") as f:
    f.write(content)

print(f"Wrote fixed ReadMe to {readme_fixed_path}")

# ------------------------------------------------------------------
# Read the catalogue using the ReadMe to interpret fixed-width columns
# ------------------------------------------------------------------
if os.path.exists(readme_path):
    table = ascii_io.read(dat_path, readme=readme_fixed_path, format="cds")
else:
    # Fallback: no ReadMe found -- try a plain whitespace-delimited read.
    # This will likely fail or mis-parse for real VizieR files; you really
    # want the ReadMe. Included only so the script doesn't hard-crash.
    print(f"WARNING: no ReadMe found at {readme_path}. "
          f"Falling back to a naive whitespace-delimited read, "
          f"which is unreliable for CDS fixed-width files.")
    table = ascii_io.read(dat_path, format="basic")

# Convert to a pandas DataFrame for everything downstream
df = table.to_pandas()

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(df.columns.tolist())
print(df.head())

# Save a plain CSV version too, so you never have to re-parse the fixed-width
# format again -- just load this CSV directly next time.
csv_path = os.path.join(DATA_DIR, "catalogue_parsed.csv")
df.to_csv(csv_path, index=False)
print(f"Saved parsed table to {csv_path}")

In [ ]:

# ------------------------------------------------------------------
# Quick visual sanity checks
# ------------------------------------------------------------------
# Guess which columns are mass, period, and semimajor axis based on usual names
candidate_mass_cols = [c for c in df.columns if "mass" in c.lower() or "sini" in c.lower()]
candidate_period_cols = [c for c in df.columns if "per" in c.lower()]
candidate_sma_cols = [c for c in df.columns if c.lower() in ("a", "sma", "axis")]
# print the ones that contain the keywords, so the user can check if they are correct
print("\nGuessed mass column(s):", candidate_mass_cols)
print("Guessed period column(s):", candidate_period_cols)
print("Guessed semimajor-axis column(s):", candidate_sma_cols)
#select as mass, period and position the first guessed column, if any, otherwise None
mass_col = candidate_mass_cols[0] if candidate_mass_cols else None
period_col = candidate_period_cols[0] if candidate_period_cols else None
sma_col = candidate_sma_cols[0] if candidate_sma_cols else None

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Mass vs period/semimajor axis ---
ax = axes[0]
x_col = sma_col or period_col
if mass_col and x_col:
    ax.scatter(df[x_col], df[mass_col], s=15, alpha=0.6, color="steelblue")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(x_col)
    ax.set_ylabel(mass_col)
    ax.set_title("Mass vs. orbital distance")
    ax.grid(True, which="both", ls=":", alpha=0.4)
else:
    ax.text(0.5, 0.5, "Set mass_col / x_col manually", ha="center", va="center")

# --- Histogram of masses ---
ax = axes[1]
if mass_col:
    ax.hist(df[mass_col].dropna(), bins=30, color="firebrick", alpha=0.7)
    #ax.set_xscale("log")
    ax.set_xlabel(mass_col)
    ax.set_ylabel("Number of planets")
    ax.set_title("Mass distribution")
else:
    ax.text(0.5, 0.5, "Set mass_col manually", ha="center", va="center")

fig.tight_layout()
fig.savefig(os.path.join(DATA_DIR, "cls_quicklook.png"), dpi=200)

In [ ]:
"""
DEFINITIONS
-----------
Super-Earth: a < 1 AU AND 1 <= Msini [Mearth] <= 20
    (table3.dat/planets.dat only give Msini, not radius)
Cold giant:  a > 1 AU AND Msini > 1 Mjup

IMPORTANT ABOUT THE CATALOG 
------------------------
In table3.dat, the per-row `CPS` field is a PER-PLANET identifier that
already includes the literature planet letter (e.g. "HD 104067 b"), same
as `Name` -- it is NOT a bare per-star ID like in table2.dat. So grouping
planets into systems by raw CPS would put every planet in its own
one-planet "system". This script instead derives the host-star ID by
stripping the trailing planet letter from `Name` (see
derive_host_from_name), and groups on that.
"""
# --------------------------------------------------------------------------
# CDS reading (astropy)
# --------------------------------------------------------------------------

def fix_readme(readme_path: Path, readme_fixed_path: Path = None) -> Path:
    """Some CDS ReadMes omit the space after the colon in
    'Byte-by-byte Description of file:XXX' (seen in this catalog for
    table7.dat, apf50.dat, planets.dat), which trips up astropy's CDS
    parser. This patches every occurrence generically."""
    readme_path = Path(readme_path)
    if readme_fixed_path is None:
        readme_fixed_path = readme_path.parent / (readme_path.name + "_fixed")
    else:
        readme_fixed_path = Path(readme_fixed_path)

    content = readme_path.read_text()
    content = re.sub(
        r"(Byte-by-byte Description of file:)(?=\S)",
        r"\1 ",
        content,
    )
    readme_fixed_path.write_text(content)
    return readme_fixed_path

def read_cds_table(dat_path, readme_path) -> pd.DataFrame:
    """Read a VizieR/CDS fixed-width .dat file via its ReadMe with astropy,
    return a pandas DataFrame with whitespace stripped from string cols.
    Falls back to a .csv read if dat_path already ends in .csv (e.g. you
    already ran df.to_csv(...) once and want to reuse that)."""
    dat_path = Path(dat_path)

    if dat_path.suffix.lower() == ".csv":
        df = pd.read_csv(dat_path)
    else:
        readme_fixed = fix_readme(Path(readme_path))
        table = ascii_io.read(str(dat_path), readme=str(readme_fixed), format="cds")
        df = table.to_pandas()

    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
    return df

In [ ]:

# --------------------------------------------------------------------------
# Classification
# --------------------------------------------------------------------------

def derive_host_from_name(name: str) -> str:
    """Strip a trailing single-letter planet designation (b, c, d, ...) to
    recover the host star ID, e.g. 'HD 104067 b' -> 'HD 104067'"""
    s = str(name).strip()
    m = re.match(r"^(.*?)\s*([b-z])$", s)
    if m and len(m.group(1)) > 0:
        return m.group(1).strip()
    return s


def classify(planets: pd.DataFrame) -> pd.DataFrame:
    p = planets.copy()
    p["Msini_Mearth"] = p["Msini"] * u.M_jup.to(u.M_earth)
    p["is_super_earth"] = (p["Msini_Mearth"] >= 1.0) & (p["Msini_Mearth"] <= 30.0)
    p["is_cold_giant"] =  (p["Msini_Mearth"] > 30.0)

    if "Name" in p.columns and p["Name"].notna().any():
        p["Host"] = p["Name"].apply(derive_host_from_name)
    else:
        p["Host"] = p["CPS"]

    return p


def normalize_id(s: str) -> str:
    """Normalize a star identifier for merging across tables that may
    differ in spacing/casing, e.g. 'HD 104067' vs 'HD104067'."""
    return re.sub(r"\s+", "", str(s)).upper()


def flag_hosts_with_cold_giant(p: pd.DataFrame) -> pd.DataFrame:
    hosts_with_cg = set(p.loc[p["is_cold_giant"], "Host"].unique())
    p = p.copy()
    p["host_has_cold_giant"] = p["Host"].isin(hosts_with_cg)
    return p


def filter_planets_dat_status(df: pd.DataFrame) -> pd.DataFrame:
    """If reading planets.dat (330 rows, includes false positives/activity/
    stellar companions flagged via Status), keep only real planets:
    K = known planet, C = new candidate."""
    if "Status" in df.columns:
        df = df[df["Status"].astype(str).str.strip().isin(["K", "C"])].copy()
    if "Name" not in df.columns:
        npl = df["Npl"].astype(str) if "Npl" in df.columns else ""
        df["Name"] = df["CPS"].astype(str) + " " + npl
    return df

In [ ]:
# --------------------------------------------------------------------------
# Statistics
# --------------------------------------------------------------------------

def describe_group(x: pd.Series, label: str) -> dict:
    x = x.dropna()
    return {
        "group": label,
        "n": len(x),
        "mean": x.mean() if len(x) else np.nan,
        "median": x.median() if len(x) else np.nan,
        "std": x.std() if len(x) else np.nan,
        "min": x.min() if len(x) else np.nan,
        "q25": x.quantile(0.25) if len(x) else np.nan,
        "q75": x.quantile(0.75) if len(x) else np.nan,
        "max": x.max() if len(x) else np.nan,
    }


In [ ]:
df = read_cds_table(dat_path, readme_path)
planets = classify(df)
### position vs mass plot of all the planets in the catalogue, highlighting super-Earths in purple and giants in blue, rest in grey
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(planets.loc[~planets["is_super_earth"] & ~planets["is_cold_giant"], "a"], planets.loc[~planets["is_super_earth"] & ~planets["is_cold_giant"], "Msini_Mearth"], s=15, alpha=0.6, color="grey", label="Other planets")

# connect SE-CG pairs within the same system
hosts_with_both = set(planets.loc[planets["is_super_earth"], "Host"]) & set(planets.loc[planets["is_cold_giant"], "Host"])

for i, host in enumerate(hosts_with_both):
    se_rows = planets[(planets["Host"] == host) & (planets["is_super_earth"])]
    cg_rows = planets[(planets["Host"] == host) & (planets["is_cold_giant"])]
    for _, se_row in se_rows.iterrows():
        for _, cg_row in cg_rows.iterrows():
            ax.plot([se_row["a"], cg_row["a"]], [se_row["Msini_Mearth"], cg_row["Msini_Mearth"]],
                    color="gray", alpha=0.4, lw=1, zorder=0,
                    label="Same system" if i == 0 else None)

ax.scatter(planets.loc[planets["is_super_earth"], "a"], planets.loc[planets["is_super_earth"], "Msini_Mearth"], s=15, alpha=0.6, color=COLORS["super_earth"], label="Super-Earths", zorder=2)
ax.scatter(planets.loc[planets["is_cold_giant"], "a"], planets.loc[planets["is_cold_giant"], "Msini_Mearth"], s=15, alpha=0.6, color=COLORS["cold_giant"], label="Cold giants", zorder=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Semi-major axis, a [AU]")
ax.set_ylabel("Msini [Mearth]")
ax.set_title("Planet mass vs. orbital distance")
ax.legend()
fig.savefig("figures/CLS/planet_catalogue.png", dpi=150)

In [ ]:
planets

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(6, 6))
super_earths = planets[planets["is_super_earth"]]
gas_giants = planets[planets["is_cold_giant"]]
axs.scatter(super_earths['a'], super_earths['Msini']/(super_earths['a']**2),  color=COLORS["super_earth"], label="Super-Earths", zorder=2)
axs.scatter(gas_giants['a'], gas_giants['Msini']/(gas_giants['a']**2),  color=COLORS["cold_giant"], label="Cold giants", zorder=2)

# connect SE-CG pairs within the same system
hosts_with_both = set(planets.loc[planets["is_super_earth"], "Host"]) & set(planets.loc[planets["is_cold_giant"], "Host"])

for i, host in enumerate(hosts_with_both):
    se_rows = planets[(planets["Host"] == host) & (planets["is_super_earth"])]
    cg_rows = planets[(planets["Host"] == host) & (planets["is_cold_giant"])]
    for _, se_row in se_rows.iterrows():
        for _, cg_row in cg_rows.iterrows():
            axs.plot([se_row["a"], cg_row["a"]], [se_row["Msini"]/(se_row["a"]**2), cg_row["Msini"]/(cg_row["a"]**2)],
                    color="gray", alpha=0.4, lw=1, zorder=0,
                    label="Same system" if i == 0 else None)


axs.set_xscale('log')
axs.set_yscale('log')
axs.set_xlabel('$a_{\\rm p}$ [AU]', fontsize=25, labelpad=20)
axs.set_ylabel('$M_{\\rm p}/a_{\\rm p}^2$ [$M_{\oplus}$/AU$^2$]', fontsize=25, labelpad=20)
axs.set_title("CLS catalogue", fontsize=15, pad=20)
plt.savefig("figures/CLS/planets_M_a2.png", bbox_inches='tight', dpi=150)


In [ ]:
from scipy.stats import spearmanr


### position vs mass plot (left) + SE distance vs CG mass (middle) + SE position vs CG position (right)
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# --- left panel: existing mass vs. distance plot ---
hosts_with_both = set(planets.loc[planets["is_super_earth"], "Host"]) & set(planets.loc[planets["is_cold_giant"], "Host"])

se_a_pairs = []   # SE orbital distance, for each SE-CG pair
cg_mass_pairs = []  # that pair's CG mass
cg_a_pairs = []  # that pair's CG orbital distance
pair_hosts = []

for i, host in enumerate(hosts_with_both):
    se_rows = planets[(planets["Host"] == host) & (planets["is_super_earth"])]
    cg_rows = planets[(planets["Host"] == host) & (planets["is_cold_giant"])]
    for _, se_row in se_rows.iterrows():
        for _, cg_row in cg_rows.iterrows():
            ax[0].plot([se_row["a"], cg_row["a"]], [se_row["Msini_Mearth"], cg_row["Msini_Mearth"]],
                       color="gray", alpha=0.4, lw=1, zorder=0,
                       label="Same system" if i == 0 else None)
            se_a_pairs.append(se_row["a"])
            cg_mass_pairs.append(cg_row["Msini_Mearth"])
            cg_a_pairs.append(cg_row["a"])
            pair_hosts.append(host)

ax[0].scatter(planets.loc[planets["is_super_earth"], "a"], planets.loc[planets["is_super_earth"], "Msini_Mearth"], s=15, alpha=0.6, color=COLORS["super_earth"], label="Super-Earths", zorder=2)
ax[0].scatter(planets.loc[planets["is_cold_giant"], "a"], planets.loc[planets["is_cold_giant"], "Msini_Mearth"], s=15, alpha=0.6, color=COLORS["cold_giant"], label="Cold giants", zorder=2)
ax[0].set_xscale("log")
ax[0].set_yscale("log")
ax[0].set_xlabel("$a_{\\rm planet}$ [AU]")
ax[0].set_ylabel("Msini [$M_{\oplus}$]")
ax[0].set_title("Planet mass vs. orbital distance")
ax[0].legend()

# --- middle panel: giant companion mass vs. SE orbital distance ---
ax[1].scatter(se_a_pairs, cg_mass_pairs, s=25, alpha=0.7, color=COLORS["super_earth"])
ax[1].set_xscale("log")
ax[1].set_yscale("log")
ax[1].set_xlabel("$a_{\\rm SE}$ [AU]")
ax[1].set_ylabel("Giant companion Msini [$M_{\oplus}$]")
ax[1].set_title("Giant companion mass vs. SE position")

# --- right panel: giant companion position vs. SE position ---
ax[2].scatter(se_a_pairs, cg_a_pairs, s=25, alpha=0.7, color=COLORS["cold_giant"])
ax[2].set_xscale("log")
ax[2].set_yscale("log")
ax[2].set_xlabel("$a_{\\rm SE}$ [AU]")
ax[2].set_ylabel("$a_{\\rm CG}$ [AU]")
ax[2].set_title("Giant companion position vs. SE position")

if len(se_a_pairs) >= 3:
    rho, pval = spearmanr(se_a_pairs, cg_mass_pairs)  # order doesn't matter for spearmanr itself
    print(f"Spearman ρ = {rho:.2f}\np = {pval:.3f}")


fig.tight_layout()
fig.savefig("figures/CLS/planet_catalogue.png", dpi=150)

In [ ]:
se = planets[planets["is_super_earth"]].copy()
se_with_cg = se[se["host_has_cold_giant"]]
se_without_cg = se[~se["host_has_cold_giant"]]

fig, ax = plt.subplots(figsize=(7, 5))
bins = np.linspace(0, 1, 21)

ax.hist(se_without_cg["a"], bins=bins, alpha=0.4, density=True,
        label=f"SE without cold giant (n={len(se_without_cg)})", color=COLORS["super_earth"])
ax.hist(se_with_cg["a"], bins=bins, alpha=0.4, density=True,
        label=f"SE with cold giant (n={len(se_with_cg)})", color="#584EC4")

x_grid = np.linspace(0, 1, 300)
# band width of 0.15 seems to give a good balance between smoothness and detail for this dataset
kde_without = gaussian_kde(se_without_cg["a"], bw_method=0.08)  
kde_with = gaussian_kde(se_with_cg["a"], bw_method=0.4)


ax.plot(x_grid, kde_without(x_grid), color=COLORS["super_earth"], lw=2)
ax.plot(x_grid, kde_with(x_grid), color="#584EC4", lw=2)

ax.set_xlabel("$a_{\\rm planet}$ [AU]")
ax.set_ylabel("Density")
ax.set_title("Super-Earth orbital distance, by presence of a cold giant")
ax.legend()
fig.tight_layout()
fig.savefig("figures/CLS/se_semimajor_axis_hist.png", dpi=150)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
bins = np.linspace(1, 20, 20)
ax.hist(without_cg["Msini_Mearth"], bins=bins, alpha=0.6, label=f"No cold giant (n={len(without_cg)})", color=COLORS["super_earth"])
ax.hist(with_cg["Msini_Mearth"], bins=bins, alpha=0.6, label=f"With cold giant (n={len(with_cg)})", color="#584EC4")
ax.set_xlabel(r"M sin i [M$_\oplus$]")
ax.set_ylabel("Number of super-Earths")
ax.set_title("Super-Earth mass, by presence of a cold giant")
ax.legend()
fig.tight_layout()
fig.savefig("figures/CLS/se_mass_hist.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].boxplot([without_cg["a"].dropna(), with_cg["a"].dropna()],tick_labels=["without CG", "With CG"])
axes[0].set_ylabel("a [AU]")
axes[0].set_title("Semi-major axis")
axes[1].boxplot([without_cg["Msini_Mearth"].dropna(), with_cg["Msini_Mearth"].dropna()],tick_labels=["Without CG", "With CG"])
axes[1].set_ylabel(r"M sin i [M$_\oplus$]")
axes[1].set_title("Mass")
fig.suptitle("Super-Earth properties: systems with vs. without a cold giant")
fig.tight_layout()
fig.savefig("figures/CLS/se_boxplots.png", dpi=150)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

violin_df = pd.concat([pd.DataFrame({"group": "Without CG","a": without_cg["a"].dropna().to_numpy(),"Msini_Mearth": without_cg["Msini_Mearth"].dropna().to_numpy(),}),
                       pd.DataFrame({"group": "With CG","a": with_cg["a"].dropna().to_numpy(),"Msini_Mearth": with_cg["Msini_Mearth"].dropna().to_numpy(),}),], ignore_index=True)

sns.violinplot(data=violin_df,x="group",y="a",ax=axes[0],palette=[COLORS["super_earth"], "#584EC4"],inner="box",hue="group")
sns.violinplot(data=violin_df,x="group",y="Msini_Mearth",ax=axes[1],palette=[COLORS["super_earth"], "#584EC4"],inner="box",hue="group")
axes[0].set_ylabel("a [AU]")
axes[0].set_xlabel("group")
axes[0].set_title("Semi-major axis")
axes[1].set_ylabel(r"M sin i [M$_\oplus$]")
axes[1].set_title("Mass")

fig.suptitle("Super-Earth properties: systems with vs. without a cold giant")
fig.tight_layout()
plt.savefig("figures/CLS/se_violinplots.png", dpi=150)